# Trumpet Pitch -> Note Class Conversion

This notebook converts the trumpet dataset annotations from continuous `pitch` (Hz) to discrete note `class` values (12 pitch classes).


In [ ]:
from pathlib import Path
import json
import math
import pandas as pd

# If run from this notebook folder, raw data is in ./raw
dataset_dir = Path.cwd()
raw_dir = dataset_dir / 'raw'

if not raw_dir.exists():
    # fallback if launched from repo root
    dataset_dir = Path('classnn-OriolFreixa/data/trumpet-class').resolve()
    raw_dir = dataset_dir / 'raw'

assert raw_dir.exists(), f'Could not find raw directory: {raw_dir}'
print(f'Dataset directory: {dataset_dir}')
print(f'Raw directory: {raw_dir}')


In [ ]:
NOTE_CLASSES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

def pitch_to_note_class(pitch_hz: float) -> str:
    """Map frequency in Hz to nearest equal-tempered note class."""
    if pitch_hz <= 0:
        raise ValueError(f'Pitch must be > 0 Hz, got {pitch_hz}')
    midi = round(69 + 12 * math.log2(pitch_hz / 440.0))
    class_index = midi % 12
    return NOTE_CLASSES[class_index]

# Quick sanity check on expected trumpet range
for hz in [230, 261.63, 293.66, 329.63, 392.0, 440.0, 450.0]:
    print(f'{hz:>7.2f} Hz -> {pitch_to_note_class(hz)}')


In [ ]:
def convert_csv_pitch_to_class(csv_path: Path) -> tuple[str, int]:
    """Convert one CSV from `pitch` column to `class` column, preserving row count."""
    df = pd.read_csv(csv_path)

    if 'pitch' not in df.columns:
        if 'class' in df.columns:
            return ('already_class', len(df))
        return ('skipped_no_pitch', len(df))

    # Convert each frame's pitch value to nearest note class
    classes = [pitch_to_note_class(float(v)) for v in df['pitch'].to_numpy()]

    out_df = pd.DataFrame({'class': classes})
    out_df.to_csv(csv_path, index=False)
    return ('converted', len(out_df))

csv_files = sorted(raw_dir.glob('*.csv'))
summary = {'converted': 0, 'already_class': 0, 'skipped_no_pitch': 0, 'rows_total': 0}

for csv_file in csv_files:
    status, rows = convert_csv_pitch_to_class(csv_file)
    summary[status] += 1
    summary['rows_total'] += rows

print('CSV conversion summary:')
for k, v in summary.items():
    print(f'  - {k}: {v}')


In [ ]:
# Update parameters.json so downstream pipeline treats this as a class parameter
parameters_path = raw_dir / 'parameters.json'

class_parameters = {
    'parameter_1': {
        'name': 'class',
        'type': 'class',
        'classes': NOTE_CLASSES,
        'description': 'Nearest note class from input pitch (12-TET).',
    }
}

if parameters_path.exists():
    pass

parameters_path.write_text(json.dumps(class_parameters, indent=2))
print(f'Updated: {parameters_path}')


In [ ]:
# Preview a few converted CSVs
for csv_path in sorted(raw_dir.glob('*.csv'))[:5]:
    df = pd.read_csv(csv_path)
    unique_vals = sorted(df['class'].unique().tolist()) if 'class' in df.columns else []
    print(f"{csv_path.name}: columns={list(df.columns)}, unique_class={unique_vals[:6]}{'...' if len(unique_vals)>6 else ''}")
